# Colab Inference: Tile Prediction Mask

This notebook loads a saved XGBoost model, runs inference on a chosen tile, and visualizes the predicted mask alongside a polygon rasterization and RGB preview.

## 1. Install and Import Dependencies

In [ ]:
# Colab: install dependencies
!pip -q install -r ONI-makeathon-challenge-2026-main/requirements.txt

import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
from shapely.geometry import shape
from rasterio.features import rasterize
import joblib

## 2. Load Saved Model in Colab

In [ ]:
# If the model is in Google Drive, mount it
# from google.colab import drive
# drive.mount("/content/drive")

DATA_DIR = "/content/data/makeathon-challenge"  # update if needed
MODEL_PATH = "/content/model/baseline_aef_logreg.joblib"  # update to your saved model

model = joblib.load(MODEL_PATH)
print("Loaded model:", type(model))

Note: this notebook assumes a model trained on AEF-only features. If you use the temporal XGBoost model, you must compute the extra NDVI/S1 features before inference.

## 3. Load Data Points and Preprocess

In [ ]:
# Pick a tile/year to visualize
TILE_ID = "18NWG_6_6"
YEAR = 2020

AEF_PATH = f"{DATA_DIR}/aef-embeddings/train/{TILE_ID}_{YEAR}.tiff"
TILES_GEOJSON = f"{DATA_DIR}/metadata/train_tiles.geojson"

with rasterio.open(AEF_PATH) as src:
    aef = src.read().astype(np.float32)
    aef_transform = src.transform
    aef_crs = src.crs
    aef_shape = src.shape

print("AEF shape:", aef.shape, "CRS:", aef_crs)

# Load the tile polygon
tiles = gpd.read_file(TILES_GEOJSON)
row = tiles[tiles["name"] == TILE_ID].iloc[0]
tile_geom = row.geometry

# Rasterize polygon to match AEF grid
polygon_mask = rasterize(
    [(tile_geom, 1)],
    out_shape=aef_shape,
    transform=aef_transform,
    fill=0,
    dtype="uint8",
)

print("Polygon mask sum:", polygon_mask.sum())

## 4. Run Inference on Specific Data Points

In [ ]:
# Flatten to (N, C)
channels, height, width = aef.shape
flat = aef.reshape(channels, height * width).transpose(1, 0)

# Predict probabilities and reshape to raster
proba = model.predict_proba(flat)[:, 1]
pred_mask = (proba.reshape(height, width) > 0.5).astype(np.uint8)

print("Predicted positives:", pred_mask.sum())

## 5. Generate Image From Data Polygon

In [ ]:
# Create a simple polygon visualization layer (1 inside polygon, 0 outside)
polygon_img = polygon_mask.astype(np.float32)

# Create a pseudo-RGB image from random AEF bands for context
rng = np.random.default_rng(42)
chosen = sorted(rng.choice(aef.shape[0], size=3, replace=False).tolist())

bands = aef[chosen].astype(np.float32)

def _norm(band: np.ndarray) -> np.ndarray:
    valid = band[np.isfinite(band)]
    if valid.size == 0:
        return np.zeros_like(band)
    lo, hi = np.percentile(valid, [2, 98])
    return np.clip((band - lo) / (hi - lo + 1e-6), 0, 1)

rgb = np.stack([_norm(bands[i]) for i in range(3)], axis=-1)
print("Using AEF bands:", [b + 1 for b in chosen])

## 6. Create and Visualize Prediction Mask

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb)
axes[0].set_title(f"AEF RGB (bands {', '.join(str(b + 1) for b in chosen)})")
axes[0].axis("off")

axes[1].imshow(polygon_img, cmap="gray")
axes[1].set_title("Tile Polygon Mask")
axes[1].axis("off")

axes[2].imshow(rgb)
axes[2].imshow(pred_mask, cmap="Reds", alpha=0.45)
axes[2].set_title("Prediction Mask Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()